# Bone Fracture Detection - YOLOv8 Training
This notebook is designed to run on Kaggle to train a YOLOv8 model for bone fracture detection.

In [ ]:
#!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 1. Prepare Dataset Paths
Kaggle datasets often have `data.yaml` files with relative paths that don't match the Kaggle working environment perfectly. This step automatically finds the `data.yaml` in your input datasets and generates a corrected version in `/kaggle/working/data.yaml` with exact absolute paths.

In [ ]:
import os
import glob
import yaml

base_dir = '/kaggle/input'
yaml_files = glob.glob(f'{base_dir}/**/data.yaml', recursive=True)
print("Found data.yaml files:", yaml_files)

if yaml_files:
    # Prioritize the one with YOLOv8 in the path if multiple exist
    data_yaml_path = next((y for y in yaml_files if 'yolov8' in y.lower()), yaml_files[0])
    dataset_dir = os.path.dirname(data_yaml_path)
    print(f"Using dataset directory: {dataset_dir}")
    
    with open(data_yaml_path, 'r') as f:
        data = yaml.safe_load(f)
        
    print("\nOriginal data.yaml:")
    print(data)
    
    # Fix paths to be absolute
    for split in ['train', 'val', 'test']:
        if split in data:
            # Handle if path in yaml has leading '../' or is just a folder name
            split_path = str(data[split]).replace('../', '')
            data[split] = os.path.join(dataset_dir, split_path)
            
    working_yaml = '/kaggle/working/data.yaml'
    with open(working_yaml, 'w') as f:
        yaml.dump(data, f)
    print("\nCreated corrected /kaggle/working/data.yaml with absolute paths:")
    print(data)
else:
    print("Error: data.yaml not found!")

## 2. Train the YOLOv8 Model
We use the `yolov8s.pt` (Small) model as a starting point. Feel free to use `yolov8n.pt` for faster training or `yolov8m.pt` for potentially better accuracy.

In [ ]:
from ultralytics import YOLO

# Load a pretrained YOLOv8 model
model = YOLO('yolov8s.pt')

# Train the model
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=50,             # Adjust number of epochs as needed
    imgsz=640,             # Image size
    batch=16,              # Batch size (reduce if out of memory)
    project='/kaggle/working/runs',
    name='bone_fracture_model',
    exist_ok=True
)

## 3. Validation
Evaluate the model's performance on the validation set.

In [ ]:
metrics = model.val()
print("mAP50-95:", metrics.box.map)
print("mAP50:", metrics.box.map50)

## 4. Inference on Sample Images
Run the trained model on some validation images and visualize the results.

In [ ]:
import matplotlib.pyplot as plt
import cv2
import random

# Get some validation images
val_images = glob.glob(os.path.join(data['val'], '*.jpg')) + \
             glob.glob(os.path.join(data['val'], '*.png')) + \
             glob.glob(os.path.join(data['val'], '*.jpeg'))

if val_images:
    # Pick a few random images
    samples = random.sample(val_images, min(3, len(val_images)))
    
    for img_path in samples:
        res = model.predict(source=img_path, save=False)
        res_plotted = res[0].plot()
        
        plt.figure(figsize=(8, 8))
        plt.imshow(cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB))
        plt.title(os.path.basename(img_path))
        plt.axis('off')
        plt.show()
else:
    print("No validation images found for inference.")

## 5. Download the Trained Model
The trained model weights are typically saved inside the runs directory. We'll locate the `best.pt` file and generate a download link so you can use it locally in the MediScan app.

In [ ]:
import os
import glob
from IPython.display import FileLink

# Search for best.pt in the runs directory
run_dir = '/kaggle/working/runs'
best_models = glob.glob(f'{run_dir}/**/*best.pt', recursive=True)

if best_models:
    best_model_path = best_models[0]
    print(f"Found model at: {best_model_path}")
    print("\nClick the link below to download the model. Rename it to `best_bone_model.pt` and place it in the `MediScan/bone/` directory locally.")
    display(FileLink(best_model_path))
else:
    print("Model file not found. Ensure training completed successfully.")